# Spinning plates 🧪

Dr Jo Martin has emailed in a panic: her compound screen on 96-well plate
EXP-042 came back from the plate reader as an export that won't even open in
Excel, the layout is in a separate file, and the controls are not behaving. Can
you find out whether the experiment worked? Read her email and follow the
[problem](../problems/data-analysis/spinning-plates.md).

Implement the helper functions in
[plate_reader.py](../src/pyproblems/plate_reader.py) as you reach each stage and
check them with `uv run pytest tests/test_plate_reader.py -q`; use this notebook
to call them, make plots and record what you find.

Run the setup cell first. It sets `data_dir`, the folder
`src/pyproblems/datasets/` that holds the three files for this plate:

- `plate_reader_export.txt`: the plate reader's export
- `plate_layout.csv`: the layout of the plate as designed
- `plate_run_record.json`: the run record kept alongside the run

Reading them is your first job.

In [ ]:
import json
from importlib.resources import files

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pyproblems.plate_reader import (
    join_layout,
    mark_usable,
    normalise_to_reference,
    parse_plate_reader,
    percent_of_vehicle,
    pivot_channels,
    plate_heatmap,
    plot_concentration_response,
    remap_readings,
    subtract_blank,
    summarise_replicates,
    validate_plate,
    z_prime,
)

data_dir = files("pyproblems.datasets")

## 1. The briefing

Three compounds (A, B and C) were each tested at four concentrations, with four
replicate wells per concentration, alongside three kinds of control:

| `sample_type` | What is in the well | Biomarker | Reference |
| --- | --- | --- | --- |
| `blank` | medium, no cells | background only | background only |
| `vehicle` | cells and DMSO only (uninhibited) | high | high |
| `inhibitor` | cells and a reference inhibitor | strongly reduced | high |


Each well was read in two channels. The **biomarker** is the signal of interest.
The **reference** is proportional to the amount of cellular material in the
well, however many cells happened to be seeded there.

Before you open the data, write down a prediction: if the experiment worked,
what should each control look like in each channel, and what would a convincing
concentration-response look like?

My prediction:



## 2. The instrument's revenge

Read the export as the instrument wrote it, and the run record beside it. What
would a parser have to cope with? Then implement `parse_plate_reader` and
`validate_plate` (`uv run pytest tests/test_plate_reader.py -k "parse or
validate" -q`) and call them below.

In [ ]:
# TODO: read the three files in data_dir.
# The export is text, the layout is a CSV table and the run record is JSON.
export_text = None
layout = None
run_record = None

In [ ]:
# Look at the export exactly as the instrument wrote it, and at the run record.
print(export_text[:2500])
run_record

In [ ]:
# TODO: parse the export and validate it. Which readings carry a QC flag?
readings = parse_plate_reader(export_text)
validate_plate(readings)
readings["qc_flag"].value_counts()

## 3. One well, one row

Implement `pivot_channels` and `join_layout` (`uv run pytest
tests/test_plate_reader.py -k "pivot or join" -q`), then build one
analysis-ready table: a row per well, with both channels, their QC flags and the
layout. Keep the raw readings unchanged; derived values go in new columns.

In [ ]:
# TODO: one row per well, then join the layout.
wells = pivot_channels(readings)
table = join_layout(wells, layout)
table.head()

## 4. The controls don't make sense

Compare what you measured with what you expected from the briefing. The pictures
below show the *intended layout* beside the two channels, plate-shaped with row
A at the top. Then summarise the controls by group.

The control measurements may look reasonable one at a time, yet fail to match
the wells they were supposedly in. What could explain this?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
plate_heatmap(table, "sample_type", axes[0], title="Intended layout")
plate_heatmap(table, "biomarker", axes[1], title="Biomarker", cmap="viridis")
plate_heatmap(table, "reference", axes[2], title="Reference", cmap="viridis")
plt.tight_layout()

In [ ]:
# TODO: summarise the control wells by sample_type. Do the signals look like their labels?
controls = table[table["sample_type"] != "treatment"]
controls.groupby("sample_type")[["biomarker", "reference"]].agg(
    ["mean", "std", "count"]
)

Describe what you see. Which controls behave as labelled, and which do not?



## 5. Competing explanations

Suppose each reading has been assigned to the wrong well in some systematic way.
Write each candidate as a mapping: a function that takes a layout well such as
`"C03"` and returns the well whose reading you think belongs to it.
`remap_readings` applies a mapping (`uv run pytest tests/test_plate_reader.py -k
"remap or z_prime" -q`), and `z_prime` (the standard assay-quality measure of
how well two control groups separate) may help you quantify the evidence.

Judge your hypotheses using the **controls only**: never choose by which one
makes a compound look best. Choose with the controls of odd replicate number,
then check your choice on the even ones, which the choice never saw.

In [ ]:
# TODO: add your own candidate mappings.
hypotheses = {
    "as exported": lambda well: well,
}

fig, axes = plt.subplots(
    1, len(hypotheses) + 1, figsize=(5 * (len(hypotheses) + 1), 4), squeeze=False
)
plate_heatmap(table, "sample_type", axes[0, 0], title="Intended layout")
for ax, (name, mapping) in zip(axes[0, 1:], hypotheses.items(), strict=True):
    remapped = remap_readings(table, mapping)
    plate_heatmap(remapped, "reference", ax, title=name, cmap="viridis")
plt.tight_layout()

In [ ]:
def control_evidence(wells):
    """TODO: return numbers showing how well the control signals match their labels."""
    raise NotImplementedError


# TODO: score each hypothesis on the odd replicates, then check on the even ones.
for name, mapping in hypotheses.items():
    remapped = remap_readings(table, mapping)
    odd = remapped[remapped["replicate"] % 2 == 1]
    even = remapped[remapped["replicate"] % 2 == 0]
    print(name, control_evidence(odd), control_evidence(even))

Which hypothesis do you choose, and what is the evidence? How decisive is it,
and how well does it hold up on the controls you held back?



Is there independent evidence of what happened? The lab's standard procedure is
to load plates with well A1 at the front-left of the tray. Compare that with the
run record.

In [ ]:
run_record["loading"], run_record["log"]

What does the run record say, and how well does it fit?



## 6. The experiment comes back to life

Implement `mark_usable`, `subtract_blank`, `normalise_to_reference`,
`percent_of_vehicle` and `summarise_replicates` (`uv run pytest
tests/test_plate_reader.py -k "usable or blank or normalise or vehicle or
summarise" -q`). Together they turn the corrected readings into a
percentage-of-vehicle response for each compound and concentration:

1. decide which wells you will not use, and why (your QC policy);
2. subtract the blank level in each channel;
3. divide the biomarker by the reference, so cell-loading differences cancel;
4. express wells relative to the vehicle controls;
5. summarise the replicates.

Then check the controls again after correction: do they meet reasonable QC
expectations?

In [ ]:
# TODO: set chosen to the mapping you have evidence for.
chosen = None

corrected = remap_readings(table, chosen)
processed = normalise_to_reference(subtract_blank(mark_usable(corrected)))
processed.head()

In [ ]:
# TODO: which wells were excluded, and why? How well do the corrected controls separate?
processed.loc[
    ~processed["usable"], ["well", "sample_type", "biomarker_flag", "reference_flag"]
]

Now compare four analyses of the same readings: the mapping you started with
versus the one you chose, each with the blank-subtracted biomarker and with the
reference-normalised biomarker. Which of the four would you want to show a
colleague? Which corrections did you need?

In [ ]:
def response_summary(mapping, column):
    wells = remap_readings(table, mapping)
    wells = normalise_to_reference(subtract_blank(mark_usable(wells)))
    percent = percent_of_vehicle(wells, column)
    return summarise_replicates(wells.assign(percent=percent), "percent")


fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True)
for i, (orientation, mapping) in enumerate(
    [("as exported", hypotheses["as exported"]), ("my choice", chosen)]
):
    for j, (signal, column) in enumerate(
        [("blank-subtracted", "biomarker_net"), ("normalised", "normalised")]
    ):
        summary = response_summary(mapping, column)
        plot_concentration_response(summary, axes[i, j], f"{orientation}, {signal}")
        axes[i, j].set_ylabel("% of vehicle")
plt.tight_layout()

Describe the response of each compound. Is it consistent across replicate wells?
Does it need both corrections to be visible?



## 7. Would you sign off on this?

The corrected experiment now appears successful. But the original
sample-to-measurement mapping was wrong, and your colleague wants to progress
Compound A to the next stage. Would you recommend **progressing** it,
**repeating** the experiment or **rejecting** the result?

Write a short conclusion, supported by your plots and QC findings. Consider:

- How convincingly do the controls establish what went wrong?
- Is there independent evidence of what happened?
- Do the corrected controls meet reasonable QC expectations?
- Are the responses consistent across replicate wells?
- Was your correction chosen independently of the result you hoped for?
- Does a convincing result on one plate establish biological reproducibility?
- What uncertainty remains about sample identities and instrument handling?
- What confirmatory experiment would you recommend?

My conclusion:

